# Part 3 — Machine Learning: predict no-shows and choose the model honestly


In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

scikit-learn 1.9.1 | pandas 3.1.1 | numpy 2.5.4


## Task 3.1 — Features, split and baseline

In [2]:
df = pd.read_csv("appointments_clean.csv")

X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows", "reminder_sent",
        "booking_channel", "visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

print("X has", X.shape[1], "columns:", list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print(f"No-shows in test set: {y_test.sum()} | overall no-show share: {round(y.mean() * 100, 1)}%")

baseline = (y_test == 0).mean()
print(f"Baseline accuracy (always predict 'Attended'): {baseline:.4f} ({(y_test == 0).sum()} of {len(y_test)})")

X has 11 columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Training rows: 792 | Test rows: 198
No-shows in test set: 49 | overall no-show share: 22.0%
Baseline accuracy (always predict 'Attended'): 0.7525 (149 of 198)


## Task 3.2 — Logistic regression

In [3]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit on training rows only
X_test_s = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_s, y_train)

pred_lr = log_reg.predict(X_test_s)
prob_lr = log_reg.predict_proba(X_test_s)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, pred_lr), 3))
print("Confusion matrix [[TN, FP], [FN, TP]]:\n", confusion_matrix(y_test, pred_lr))
print("Precision:", round(precision_score(y_test, pred_lr), 3))
print("Recall   :", round(recall_score(y_test, pred_lr), 3))
print("F1       :", round(f1_score(y_test, pred_lr), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, prob_lr), 3))

Accuracy : 0.788
Confusion matrix [[TN, FP], [FN, TP]]:
 [[139  10]
 [ 32  17]]
Precision: 0.63
Recall   : 0.347
F1       : 0.447
ROC-AUC  : 0.789


In [4]:
coefs = pd.DataFrame({"feature": X.columns, "coefficient": log_reg.coef_[0]})
print(coefs.round(3).to_string(index=False))

i = list(X.columns).index("reminder_sent")
coef_rem = log_reg.coef_[0][i]
sd_rem = scaler.scale_[i]
print(f"\nreminder_sent coefficient (scaled): {coef_rem:.3f}")
print(f"np.exp(coefficient)            = {np.exp(coef_rem):.2f}  -> odds multiplier per 1 SD (SD = {sd_rem:.2f})")
print(f"np.exp(coefficient / scale_)   = {np.exp(coef_rem / sd_rem):.2f}  -> odds multiplier for No (0) -> Yes (1)")
print(f"i.e. odds of a no-show about {round((1 - np.exp(coef_rem / sd_rem)) * 100)}% lower with a reminder")

                feature  coefficient
                    age       -0.254
            distance_km        0.253
                fee_inr        0.453
              lead_days        0.614
         prior_no_shows        0.650
          reminder_sent       -0.581
    booking_channel_App       -0.047
  booking_channel_Phone        0.216
booking_channel_Walk-In       -0.186
visit_type_Consultation        0.016
   visit_type_Follow-up       -0.016

reminder_sent coefficient (scaled): -0.581
np.exp(coefficient)            = 0.56  -> odds multiplier per 1 SD (SD = 0.49)
np.exp(coefficient / scale_)   = 0.31  -> odds multiplier for No (0) -> Yes (1)
i.e. odds of a no-show about 69% lower with a reminder


**Why the two numbers differ:** the model was fitted on standardised features, so `np.exp(coef)` is the change in
odds for a one-*standard-deviation* step in `reminder_sent` (SD ≈ 0.49, about half of the real step), whereas the
real No → Yes change is a step of 1, i.e. about two standard deviations — so the coefficient must be divided by
`scaler.scale_` before exponentiating.

**Conclusion (association, not cause):** in this model, holding the other features constant, appointments with a
reminder sent have substantially lower odds of a no-show (the No → Yes odds multiplier printed above). The data are
observational — reminders were not assigned at random — so this does not prove that sending reminders *causes*
fewer no-shows.

## Task 3.3 — Decision tree and overfitting

In [5]:
tree_full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree_3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

for name, m in [("Full-depth tree", tree_full), ("Depth-3 tree", tree_3)]:
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test, m.predict(X_test))
    print(f"{name:<16} depth={m.get_depth():<3} train acc={tr:.3f}  test acc={te:.3f}  gap={tr - te:.3f}")
print(f"Baseline test accuracy: {baseline:.4f}")

Full-depth tree  depth=15  train acc=1.000  test acc=0.672  gap=0.328
Depth-3 tree     depth=3   train acc=0.822  test acc=0.778  gap=0.044
Baseline test accuracy: 0.7525


**What the gap shows:** the unlimited tree fits the training set perfectly but does worse on unseen bookings (below
even the always-"Attended" baseline), so it has memorised noise — it overfits; limiting the depth to 3 gives up
training accuracy but generalises much better, with a small train/test gap.

## Task 3.4 — Random forest and a comparison table

In [6]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
pred_rf = rf.predict(X_test)
prob_rf = rf.predict_proba(X_test)[:, 1]

print("Training accuracy:", round(accuracy_score(y_train, rf.predict(X_train)), 3))
print("Test accuracy    :", round(accuracy_score(y_test, pred_rf), 3))
print("Recall           :", round(recall_score(y_test, pred_rf), 3))
print("ROC-AUC          :", round(roc_auc_score(y_test, prob_rf), 3))

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop three feature importances:")
print(importances.head(3).round(3))

Training accuracy: 1.0
Test accuracy    : 0.747


Recall           : 0.306
ROC-AUC          : 0.706

Top three feature importances:
lead_days      0.220
age            0.197
distance_km    0.195
dtype: float64


In [7]:
comparison = pd.DataFrame([
    {"model": "Logistic regression", "accuracy": accuracy_score(y_test, pred_lr),
     "recall": recall_score(y_test, pred_lr), "roc_auc": roc_auc_score(y_test, prob_lr)},
    {"model": "Decision tree (depth 3)", "accuracy": accuracy_score(y_test, tree_3.predict(X_test)),
     "recall": recall_score(y_test, tree_3.predict(X_test)),
     "roc_auc": roc_auc_score(y_test, tree_3.predict_proba(X_test)[:, 1])},
    {"model": "Random forest", "accuracy": accuracy_score(y_test, pred_rf),
     "recall": recall_score(y_test, pred_rf), "roc_auc": roc_auc_score(y_test, prob_rf)},
]).round(3)
comparison

,model,accuracy,recall,roc_auc
0,Logistic regression,0.788,0.347,0.789
1,Decision tree (depth 3),0.778,0.224,0.700
2,Random forest,0.747,0.306,0.706


**Which looks best (for information only):** logistic regression has the highest test ROC-AUC, i.e. it ranks
risky bookings above safe ones most reliably, which is what a "whom to phone first" list needs; recall at the
default 0.5 threshold is low for all three models, which is why the threshold is revisited in Task 3.5. This table
uses the test set, so it is **not** used to choose the model — the formal choice below uses cross-validation on
the training set only.

## Task 3.5 — Tune, choose the model and the threshold, then test once

In [8]:
# (a) grid search on the training set (cross-validated ROC-AUC)
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
                    cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("best_params_:", grid.best_params_)
print("best_score_ (cross-validated ROC-AUC):", round(grid.best_score_, 3))

best_params_: {'max_depth': 3, 'n_estimators': 200}
best_score_ (cross-validated ROC-AUC): 0.767


In [9]:
# (b) choose the model from 5-fold cross-validated ROC-AUC on the training set only
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

candidates = {
    "Logistic regression": lr_pipe,   # unscaled X_train: the pipeline re-fits the scaler in every fold
    "Decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=42),
}
cv_scores = {name: cross_val_score(m, X_train, y_train, cv=5, scoring="roc_auc").mean()
             for name, m in candidates.items()}
cv_scores["Tuned random forest"] = grid.best_score_
candidates["Tuned random forest"] = RandomForestClassifier(random_state=42, **grid.best_params_)

cv_table = pd.Series(cv_scores, name="cv_roc_auc").round(3).sort_values(ascending=False)
print(cv_table.to_string())

winner_name = cv_table.index[0]
winner = candidates[winner_name]
print("\nWinner (highest cross-validated training ROC-AUC):", winner_name)

Logistic regression        0.774
Tuned random forest        0.767
Random forest              0.747
Decision tree (depth 3)    0.678

Winner (highest cross-validated training ROC-AUC): Logistic regression


In [10]:
# (c) choose the threshold from out-of-fold training probabilities
oof = cross_val_predict(winner, X_train, y_train, cv=5, method="predict_proba")[:, 1]

rows = []
for t in [0.5, 0.35]:
    flag = (oof >= t).astype(int)
    rows.append({"threshold": t, "precision": precision_score(y_train, flag),
                 "recall": recall_score(y_train, flag), "f1": f1_score(y_train, flag),
                 "flagged": int(flag.sum())})
thr_table = pd.DataFrame(rows).round(3)
print(thr_table.to_string(index=False))

THRESHOLD = thr_table.loc[thr_table["f1"].idxmax(), "threshold"]
print("\nChosen threshold (higher out-of-fold F1):", THRESHOLD)

 threshold  precision  recall    f1  flagged
      0.50      0.575   0.249 0.347       73
      0.35      0.513   0.467 0.489      154

Chosen threshold (higher out-of-fold F1): 0.35


**Why recall matters here:** a missed warning (a no-show the model did not flag) costs an empty doctor slot, while a
false alarm only costs the front desk a one-minute phone call — so catching more of the real no-shows is worth some
extra calls, and the lower threshold, which has the higher F1 *and* much higher recall, is preferred.

In [11]:
# (d) fit the winner on the full training set and evaluate ONCE on the test set
winner.fit(X_train, y_train)          # for the pipeline, this fits its own scaler on X_train
risk_prob = winner.predict_proba(X_test)[:, 1]
test_flag = (risk_prob >= THRESHOLD).astype(int)

print(f"Final model: {winner_name} at threshold {THRESHOLD}")
print("Precision:", round(precision_score(y_test, test_flag), 3))
print("Recall   :", round(recall_score(y_test, test_flag), 3))
print("Flagged  :", test_flag.sum(), "of", len(y_test), "test bookings")
print("ROC-AUC  :", round(roc_auc_score(y_test, risk_prob), 3))

Final model: Logistic regression at threshold 0.35
Precision: 0.571
Recall   : 0.571
Flagged  : 49 of 198 test bookings
ROC-AUC  : 0.789


In [12]:
# (e) save the test-set risk scores
scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": risk_prob.round(4),
    "flag": test_flag,
})
scores.to_csv("test_risk_scores.csv", index=False)
print("Saved test_risk_scores.csv:", scores.shape, "| rows with flag = 1:", scores["flag"].sum())
scores.sort_values("risk_prob", ascending=False).head(10)

Saved test_risk_scores.csv: (198, 4) | rows with flag = 1: 49


,appointment_id,no_show,risk_prob,flag
68,A0635,1,0.9372,1
24,A0544,1,0.9241,1
141,A0293,1,0.9196,1
106,A0950,1,0.8186,1
114,A0996,0,0.7667,1
69,A0067,0,0.7498,1
172,A0769,1,0.7444,1
10,A0364,0,0.7319,1
160,A0806,1,0.7035,1
173,A0560,0,0.7019,1


**How a clinic would really choose the threshold:** start from how many follow-up calls the front desk can make in an
evening and set the threshold so that the flagged list fits that capacity, weighing the cost of each empty slot
against the cost of each extra call — 0.35 here is a reasonable default, not a fixed rule.